In [2]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/project-1/mango-disease-prediction"
)

DATA_SRC_DIR = PROJECT_ROOT / "src" / "data"

DATA_SRC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Folder:", DATA_SRC_DIR)
print("Exists:", DATA_SRC_DIR.exists())

Folder: /content/drive/MyDrive/project-1/mango-disease-prediction/src/data
Exists: True


In [4]:
init_file = DATA_SRC_DIR / "__init__.py"

init_file.touch()

print("Created:", init_file)

Created: /content/drive/MyDrive/project-1/mango-disease-prediction/src/data/__init__.py


Now create dataset.py

In [5]:
%%writefile /content/drive/MyDrive/project-1/mango-disease-prediction/src/data/dataset.py

import torch

from torch.utils.data import Dataset

from PIL import Image


class MangoDiseaseDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_transform=None,
        environmental_data=None
    ):
        """
        Dataset for mango disease classification.

        Each sample contains:
            - image
            - environmental features
            - disease label
        """

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.image_transform = image_transform

        self.environmental_data = environmental_data


    def __len__(self):
        """
        Return the number of samples.
        """

        return len(self.dataframe)


    def __getitem__(self, index):
        """
        Load and return one sample.
        """

        # ==========================================
        # LOAD IMAGE
        # ==========================================

        image_path = self.dataframe.iloc[
            index
        ]["image_path"]

        image = Image.open(
            image_path
        ).convert("RGB")


        # Apply image preprocessing
        if self.image_transform is not None:

            image = self.image_transform(
                image
            )


        # ==========================================
        # ENVIRONMENTAL DATA
        # ==========================================

        if self.environmental_data is not None:

            environment = self.environmental_data[
                index
            ]

            if not isinstance(
                environment,
                torch.Tensor
            ):

                environment = torch.tensor(
                    environment,
                    dtype=torch.float32
                )

        else:

            environment = torch.empty(
                0,
                dtype=torch.float32
            )


        # ==========================================
        # LABEL
        # ==========================================

        label = self.dataframe.iloc[
            index
        ]["label"]

        label = torch.tensor(
            label,
            dtype=torch.long
        )


        # ==========================================
        # RETURN
        # ==========================================

        return {
            "image": image,
            "environment": environment,
            "label": label
        }

Writing /content/drive/MyDrive/project-1/mango-disease-prediction/src/data/dataset.py


In [6]:
print("Files in src/data:")

for item in DATA_SRC_DIR.iterdir():
    print(item.name)

Files in src/data:
dataset.py
__init__.py


Test importing it

In [7]:
import sys

PROJECT_ROOT_STR = str(PROJECT_ROOT)

if PROJECT_ROOT_STR not in sys.path:
    sys.path.append(PROJECT_ROOT_STR)

In [8]:
from src.data.dataset import MangoDiseaseDataset

print("✅ MangoDiseaseDataset imported successfully!")

✅ MangoDiseaseDataset imported successfully!
